# Curriculum 05 · Lab 3 — Decomposition: split multi-hop questions into single-hop chunks

**Goal:** Make retrieval multi-hop-capable. A question like "Who wrote the
song, and when did that writer die?" needs facts from 2+ chunks. Plain top-k
embeds the WHOLE question and returns the single most-similar chunk — the
other fact's chunk is out of reach before any LLM reads anything.

```
Transformer : DecomposeRetriever (retrieval/decompose.py)
LLM         : llama-3.3-70b-versatile (GroqLLM) — the decomposer, never the embedder
Flow        : original question + N sub-questions -> retrieve each -> merge & dedupe
Data        : HotpotQA — 3 multi-hop questions with self-contained 10-paragraph
              contexts and gold supporting_facts labels
Embedding   : BGE (BAAI/bge-base-en-v1.5, local, CPU)
```

**Why decompose:** each fact gets its own retrieval pass, so every fact's
chunk enters the candidate set. The gold `supporting_facts` labels make the
lesson measurable: every question here is one plain top-3 retrieval could
NOT answer (verified), and decomposition brings all gold paragraphs back.

This is the third lab of track 05-query-transformation (see
`.omo/plans/layer1-rag-playbook.md`).


## 0 · Setup — environment, imports & repo paths

**WHAT:** Installs the lab's dependencies (a no-op if already present),
loads `GROQ_API_KEY` from the repo-root `.env`, and puts the repo-root
component library on `sys.path` so this notebook reuses `retrieval/*.py`,
`llms/groq.py`, `vectordb/faiss.py` and `embeddings/bge.py` exactly like the
lab script.

**WHY:** Everything embeds **locally** with BGE via sentence-transformers —
no API embeddings anywhere. The LLM is only the query-*transformation* step
(Groq's `llama-3.3-70b-versatile`; a commented Gemini alternative is kept in
the source). The retriever classes live in the repo's shared component
library (`retrieval/`), not inside the lab, so the exact same code path runs
here, in the `.py`, and in later tracks.

**Paths:** the next cell resolves the **repo root** automatically — it works
whether the kernel launches from the repo root (like the lab script) or from
the notebook's own folder (the Jupyter default) — and `cd`s into it so every
path stays repo-relative.

**WHAT TO EXPECT:** no output from the pip cell (packages already
installed), a silent import from the second. The BGE model is loaded lazily
when the experiment cell first calls it; the Groq key is read from `.env`.


In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings (embeddings/bge.py)
#   faiss-cpu             -> the FAISS index (vectordb/faiss.py)
#   langchain-groq        -> GroqLLM (the decomposer LLM, llms/groq.py)
#   python-dotenv         -> loads GROQ_API_KEY from the repo-root .env
%pip install sentence-transformers faiss-cpu langchain-groq python-dotenv



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import json
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")  # GROQ_API_KEY lives in the repo-root .env

from embeddings.bge import BGEEmbedding  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from llms.groq import GroqLLM  # noqa: E402
from retrieval.decompose import DecomposeRetriever  # noqa: E402
from retrieval.similarity import SimilarityRetriever  # noqa: E402
from vectordb.faiss import FAISSVectorStore  # noqa: E402


## 1 · Configuration — the experiment's knobs

**WHAT:** `HOTPOTQA_PATH` (the benchmark's dev file — one JSON array),
`QUESTION_IDS` (three pre-selected multi-hop questions whose plain top-3
misses a gold supporting paragraph), `TOP_K = 3` (plain depth — deliberately
small: one question can only fit one fact), `DECOMPOSE_TOP_K = 6` (merged
depth after original + sub-question retrievals), and the Groq model name.

**WHY:** The three questions are the lab's contract — each verified, before
shipping, to fail plain top-3 and recover all gold paragraphs under
decomposition. That asymmetry is the entire lesson.


In [3]:
HOTPOTQA_PATH = Path("Data/corpus/hotpotqa/hotpot_dev_distractor_v1.json")
# Three multi-hop questions, pre-selected from hotpotqa: for each one, plain
# top-3 misses a gold supporting paragraph while decomposition recovers it
# (verified against the data + the Groq decomposer before shipping).
QUESTION_IDS = [
    "5a722b8655429971e9dc9329",  # "Who was the writer of These Boots… and who died in 2007?"
    "5a8a3e745542996c9b8d5e70",  # "What is the name for the adventure in Tunnels and Trolls…?"
    "5adf37a95542995ec70e8f97",  # "The 2011–12 VCU Rams… represented VCU which was founded in what year?"
]
TOP_K = 3  # plain retrieval depth (deliberately small: the question can only fit one fact)
DECOMPOSE_TOP_K = 6  # merged depth after original + sub-question retrievals
LLM_MODEL = "llama-3.3-70b-versatile"  # Groq is the *decomposer* LLM, never the embedder
# (Gemini alternative: LLM_MODEL = "gemini-2.5-flash" — needs GOOGLE_API_KEY in .env)
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
N_PARAGRAPHS = 10  # hotpotqa provides 10 context paragraphs per question


## 2 · Load — hotpotqa questions with their self-contained contexts

**WHAT:** `load_hotpotqa` reads the JSON array and returns the requested rows;
`distinct_supporting_titles` dedupes each row's gold `supporting_facts`
titles; `preview` flattens a paragraph for one-line printing. Hotpotqa's
`context` field is 10 `[title, [sentences…]]` paragraphs per question — the
sentences are joined back into one passage per paragraph.

**WHY:** Each question ships its own 10-paragraph context, so every question
gets its own small FAISS store — self-contained, no cross-question leakage —
and the gold `supporting_facts` titles are the ground truth both retrievers
are measured against.


In [4]:
def load_hotpotqa(path: Path, ids: list[str]) -> dict[str, dict]:
    """Return {qid: question_dict} for the requested hotpotqa rows.

    Each dict keeps ``question``, ``answer``, ``supporting_facts`` (list of
    [title, sentence_index] pairs) and ``context`` (10 [title, [sentences…]]
    paragraphs). The gold ``supporting_facts`` titles are the ground truth
    the lab measures both retrievers against.
    """
    by_id: dict[str, dict] = {}
    with open(path) as f:
        for q in json.load(f):  # hotpotqa is one JSON array, not JSON-lines
            if q["_id"] in ids:
                by_id[q["_id"]] = q
    return by_id


def distinct_supporting_titles(q: dict) -> list[str]:
    """Gold supporting paragraph titles, deduplicated, in first-seen order."""
    return list(dict.fromkeys(s[0] for s in q["supporting_facts"]))


def preview(text: str, limit: int = 62) -> str:
    """Flatten a paragraph for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


## 3 · Experiment — per question: plain top-3 vs decomposed retrieval

**WHAT:** `run_experiment` loads the three questions, builds one small FAISS
store per question over its 10 paragraphs, runs plain `SimilarityRetriever`
top-3, then `DecomposeRetriever` (Groq decomposer + the same inner
retriever) — recording the plain titles, the LLM's sub-questions, the
decomposed titles, and the timings.

**WHY:** Plain and decomposed share the same store and the same inner
retriever — only the query-issuing strategy changes. That isolates the
transformation as the single cause of any coverage difference.


In [5]:
def run_experiment() -> dict:
    questions = load_hotpotqa(HOTPOTQA_PATH, QUESTION_IDS)
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME)

    results = []
    for qid in QUESTION_IDS:
        q = questions[qid]
        distinct = distinct_supporting_titles(q)

        # Each question gets its own small store over its 10 paragraphs.
        chunks = [
            Document(page_content=" ".join(sents), metadata={"title": title})
            for title, sents in q["context"]
        ]
        store = FAISSVectorStore(embedding=embedder)
        t0 = time.perf_counter()
        store.add(chunks, embeddings=embedder.embed_documents(
            [c.page_content for c in chunks]
        ))
        index_s = time.perf_counter() - t0

        inner = SimilarityRetriever(store, top_k=TOP_K)
        plain_docs = inner.retrieve(q["question"])

        decomposer = GroqLLM(model=LLM_MODEL)
        decomposed = DecomposeRetriever(decomposer, inner, top_k=DECOMPOSE_TOP_K)
        t0 = time.perf_counter()
        sub_questions = decomposed._decompose(q["question"])
        decompose_s = time.perf_counter() - t0
        decomposed_docs = decomposed.retrieve(q["question"])

        results.append(
            {
                "qid": qid,
                "question": q["question"],
                "answer": q["answer"],
                "supporting": distinct,
                "plain_titles": [d.metadata["title"] for d in plain_docs],
                "sub_questions": sub_questions,
                "decomposed_titles": [d.metadata["title"] for d in decomposed_docs],
                "decomposed_first": preview(decomposed_docs[0].page_content),
                "index_s": index_s,
                "decompose_s": decompose_s,
                "n_paragraphs": len(chunks),
            }
        )

    return {"questions": questions, "results": results}


## 4 · Run — execute the experiment

**WHAT:** Calls `run_experiment()` — 3 small embeds, 3 index builds, and one
Groq decomposition call per question (the sub-questions) take a few seconds.
The artifact dict is kept as `exp`.

**WHY:** Demo and gate both read this single `exp` — the sub-questions and
the decomposed coverage come from the same run.


In [6]:
exp = run_experiment()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 5 · Demo — read the artifact

**WHAT:** `print_demo` prints the three questions with their gold supporting
paragraphs, the plain top-3 titles (with the MISSING gold paragraphs called
out), then the decomposed path: the LLM's sub-questions and the merged
titles, marking every recovered gold paragraph.

**WHY:** The `MISSING` line is the problem, the `recovered` line is the fix —
side by side per question. Read the sub-questions as the decomposition
working: one fact per question, each answerable by a single chunk.


In [7]:
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 03 — Decomposition: split multi-hop questions into single-hop chunks")
    print(f"{BGE_MODEL_NAME} (local) | HotpotQA | {LLM_MODEL} decomposer")
    print("=" * 66)

    print(f"\n[1] Questions ({len(exp['results'])} from hotpotqa, each with its own")
    print("    10-paragraph context + gold supporting facts):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"][:8]}] "{r["question"]}"')
        print(f"      answer: {r['answer']!r}")
        print(f"      gold supporting paragraphs: {r['supporting']}")

    print(f"\n[2] Plain top-{TOP_K} (raw question, one retrieval pass):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"][:8]}] "{r["question"]}"')
        for t in r["plain_titles"]:
            mark = " [SUPPORTING]" if t in r["supporting"] else ""
            print(f"      - {t}{mark}")
        missing = [t for t in r["supporting"] if t not in r["plain_titles"]]
        print(f"      MISSING gold paragraphs: {missing if missing else 'none'}")

    print(f"\n[3] Decomposed (original + sub-questions, merged to top-{DECOMPOSE_TOP_K}):")
    for r in exp["results"]:
        print(f'\n    Q[{r["qid"][:8]}] "{r["question"]}"')
        print(f"      sub-questions ({r['decompose_s']:.1f}s):")
        for s in r["sub_questions"]:
            print(f"        - {s}")
        for t in r["decomposed_titles"]:
            mark = " [SUPPORTING]" if t in r["supporting"] else ""
            print(f"      - {t}{mark}")
        recovered = all(t in r["decomposed_titles"] for t in r["supporting"])
        print(f"      all gold paragraphs recovered: {recovered}")

    print("\n[4] Takeaway")
    print("    Decomposition fixes retrieval for multi-hop questions: one")
    print("    retrieval pass per fact (plus the original), merged. The gold")
    print("    supporting_facts labels show the win — every question here is")
    print("    one plain top-3 retrieval could NOT answer, and decomposition")
    print("    brings both paragraphs into the candidate set before the LLM")
    print("    ever reads them.")


In [8]:
print_demo(exp)


Lab 03 — Decomposition: split multi-hop questions into single-hop chunks
BAAI/bge-base-en-v1.5 (local) | HotpotQA | llama-3.3-70b-versatile decomposer

[1] Questions (3 from hotpotqa, each with its own
    10-paragraph context + gold supporting facts):

    Q[5a722b86] "Who was the writer of These Boots Are Made for Walkin' and who died in 2007?"
      answer: 'Barton Lee Hazlewood'
      gold supporting paragraphs: ["These Boots Are Made for Walkin'", 'Lee Hazlewood']

    Q[5a8a3e74] "What is the name for the adventure in "Tunnels and Trolls", a game designed by Ken St. Andre?"
      answer: 'Arena of Khazan'
      gold supporting paragraphs: ['Arena of Khazan', 'Tunnels &amp; Trolls']

    Q[5adf37a9] "The 2011–12 VCU Rams men's basketball team, led by third year head coach Shaka Smart, represented Virginia Commonwealth University which was founded in what year?"
      answer: '1838'
      gold supporting paragraphs: ["2011–12 VCU Rams men's basketball team", 'Virginia Commonwealth 

## 6 · Verification gate — the same checks the .py runs

**WHAT:** Runs the exact `verify_gate`: all 3 questions loaded with their
full 10-paragraph contexts, every question having >= 2 distinct gold
supporting paragraphs, plain top-3 MISSING at least one gold paragraph (the
problem — deterministic, no LLM), every question generating >= 1 sub-question
(none a verbatim repeat of the original), and decomposition recovering ALL
gold paragraphs with deduplicated titles (the fix).

**WHY:** `python 03-decomposition.py --verify` must print 18/18 PASS; this
cell proves the notebook reproduces the verified `.py` exactly. The recovery
gate is the strong claim: the merged retrieval provably widened the candidate
set to the facts plain retrieval could not reach.


In [9]:
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    # Structural: all requested questions loaded, with their full contexts.
    checks.append(("all 3 hotpotqa questions loaded",
                   len(exp["results"]) == len(QUESTION_IDS)))
    checks.append(("every question indexes its full 10-paragraph context",
                   all(r["n_paragraphs"] == N_PARAGRAPHS for r in exp["results"])))
    checks.append(("every question has >= 2 distinct gold supporting paragraphs",
                   all(len(r["supporting"]) >= 2 for r in exp["results"])))

    # The problem: plain top-3 must MISS at least one gold paragraph — this
    # is what makes the multi-hop question worth decomposing (deterministic:
    # pure retrieval, no LLM).
    for r in exp["results"]:
        tag = f"Q{r['qid'][:8]}"
        missing = [t for t in r["supporting"] if t not in r["plain_titles"]]
        checks.append((f"{tag} plain top-{TOP_K} misses >= 1 gold paragraph",
                       len(missing) >= 1))

    # The fix: decomposition must generate sub-questions…
    for r in exp["results"]:
        tag = f"Q{r['qid'][:8]}"
        checks.append((f"{tag} generated >= 1 sub-question",
                       len(r["sub_questions"]) >= 1))
        checks.append((f"{tag} no sub-question repeats the original verbatim",
                       all(s.strip().lower() != r["question"].strip().lower()
                           for s in r["sub_questions"])))

    # …and the merged retrieval must recover EVERY gold paragraph. This is
    # the teaching gate: it only passes when decomposition actually widened
    # the candidate set to the facts plain retrieval could not reach.
    for r in exp["results"]:
        tag = f"Q{r['qid'][:8]}"
        recovered = all(t in r["decomposed_titles"] for t in r["supporting"])
        checks.append((f"{tag} decomposition recovers ALL gold paragraphs",
                       recovered))
        checks.append((f"{tag} decomposed titles are deduplicated",
                       len(r["decomposed_titles"]) == len(set(r["decomposed_titles"]))))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


In [10]:
verify_gate(exp)


verification gate:
  [PASS] all 3 hotpotqa questions loaded
  [PASS] every question indexes its full 10-paragraph context
  [PASS] every question has >= 2 distinct gold supporting paragraphs
  [PASS] Q5a722b86 plain top-3 misses >= 1 gold paragraph
  [PASS] Q5a8a3e74 plain top-3 misses >= 1 gold paragraph
  [PASS] Q5adf37a9 plain top-3 misses >= 1 gold paragraph
  [PASS] Q5a722b86 generated >= 1 sub-question
  [PASS] Q5a722b86 no sub-question repeats the original verbatim
  [PASS] Q5a8a3e74 generated >= 1 sub-question
  [PASS] Q5a8a3e74 no sub-question repeats the original verbatim
  [PASS] Q5adf37a9 generated >= 1 sub-question
  [PASS] Q5adf37a9 no sub-question repeats the original verbatim
  [PASS] Q5a722b86 decomposition recovers ALL gold paragraphs
  [PASS] Q5a722b86 decomposed titles are deduplicated
  [PASS] Q5a8a3e74 decomposition recovers ALL gold paragraphs
  [PASS] Q5a8a3e74 decomposed titles are deduplicated
  [PASS] Q5adf37a9 decomposition recovers ALL gold paragraphs
  [PA

0